In [1]:
# parameters
p_dt=""


StatementMeta(, 2b6f36c6-e2e2-4ee9-9034-7c2b0da3a180, 3, Finished, Available, Finished, False)

# Трансофрмація даних по Terminals з Bronze в Silver.

---
Використовується в Pipeline

---


## 1. Сервісні функції

In [2]:
from pyspark.sql.functions import lit, col, expr
from delta.tables import DeltaTable
from datetime import datetime
import json
import uuid



def set_opdate_status(p_dt, p_status):
    """Встановити статус операційного дня"""
    # якщо ставимо статус "O" то всі інші треба закрити в "C"  
    if p_status=='O':
        spark.sql("""
                    UPDATE psh_dwh_lh.dbo.calendar
                    SET STATUS='C'
                    WHERE  /*OPDATE = :opdate and*/ STATUS=:status
                """, {"status": p_status})

    # Встановлюю статус поточного операційного дня
    spark.sql("""
                UPDATE psh_dwh_lh.dbo.calendar
                SET STATUS=:status
                WHERE  OPDATE = :opdate
            """, {"opdate": p_dt, "status": p_status})

def init_upload(p_opdate):
    """
        Створю запис про початок процесу завантаження даних  плвертає його upload_id
    """
    # Генеруємо унікальний ID завантаження за допомогою Python
    u_id = str(uuid.uuid4())
    # Визначаємо поточний час для позначки завантаження
    current_ts = datetime.now()
    spark.sql(
        """
            INSERT INTO  psh_dwh_lh.dbo.ev2_silver_upload_grn
            (upload_id, opdate, upload_status, upload_at)
            VALUES
            (:upload_id, :opdate, :upload_status, :upload_at )
        """, 
        { 
            "upload_id": u_id, 
            "opdate": p_dt, 
            "upload_status": 'STARTED', 
            "upload_at": current_ts 
        }
    )
    print(f"Ініційовано завантаження {u_id} для дати {p_opdate}")
    return u_id

def close_upload(p_upload_id):
    """
        Закриваю (фінішую) запис про  процесу завантаження даних по upload_id
    """

    # Визначаємо поточний час для позначки завантаження
    current_ts = datetime.now()
    spark.sql(
        """
            UPDATE  psh_dwh_lh.dbo.ev2_silver_upload_grn
                SET upload_status = :upload_status, 
                    upload_at = :upload_at
            WHERE  upload_id = :upload_id       
          
        """, 
        { 
            "upload_id": p_upload_id, 
            "upload_status": 'SUCCESS', 
            "upload_at": current_ts 
        }
    )
    print(f"Фінішовано успішно процес завантаження {p_upload_id} ")
   


StatementMeta(, 2b6f36c6-e2e2-4ee9-9034-7c2b0da3a180, 4, Finished, Available, Finished, False)

## 2. Основний обробник

### 2.1. Створюємо новий запис про початок обробки 

In [3]:
up_opdate = p_dt
continue_processing = True
result_data = {
    "status": "Error",
    "error": None,
    "upload_id": None,
    "upload_opdate": None    
}

if up_opdate == "" or not up_opdate:
    continue_processing = False
    result_data["status"] = "Error"
    result_data["error"] = "Не знайдено жодного відкритого дня OPDATE" 

if not continue_processing :
    mssparkutils.notebook.exit(json.dumps(result_data))

try:
    up_id=init_upload(up_opdate)
    result_data["upload_id"] = up_id
    result_data["upload_opdate"] = up_opdate

except Exception as e:
    print(f"Помилка обробки : {e}")
    continue_processing = False
    result_data["status"] = "Error"
    result_data["error"] = str(e)     

StatementMeta(, 2b6f36c6-e2e2-4ee9-9034-7c2b0da3a180, 5, Finished, Available, Finished, False)

Ініційовано завантаження c96f418c-b315-4bdd-90e0-ce40c73b0ac3 для дати 2025-04-15


### 2.2. Завантажуємо довідник терміналів

## 

In [4]:
if not continue_processing :
    mssparkutils.notebook.exit(json.dumps(result_data))

try:

    # Виконю merge з довідника терміналів
    print("Оновлення довідника терміналів")
    trm_up_id=result_data["upload_id"] 
    trm_idt_ts = datetime.now()
    target_table="psh_dwh_lh.dbo.ev2_silver_terminals_dim"
    source_table="psh_dwh_lh.dbo.terminals_d"
    spark.sql(f"""
            MERGE INTO {target_table} AS target
            USING {source_table} AS source
            ON target.terminal_id = source.TERMINAL_ID
            WHEN MATCHED THEN
                UPDATE SET 
                    target.terminal_name = source.TERMINAL_NAME, 
                    target.isactive = source.ISACTIVE,
                    target.upload_id=:trm_up_id,
                    target.idt = :idt_ts
                    
            WHEN NOT MATCHED THEN
                INSERT (terminal_id, terminal_name, isactive, upload_id, idt )
                VALUES ( source.TERMINAL_ID, source.TERMINAL_NAME, source.ISACTIVE, 
                         :trm_up_id, :idt_ts)
        """
        , {
            "trm_up_id": trm_up_id, 
            "idt_ts":trm_idt_ts 
        }
    )
    print("Оновлення довідника терміналів-OK")
except Exception as e:
    print(f"Оновлення довідника терміналів ERROR : {e}")
    continue_processing = False
    result_data["status"] = "Error"
    result_data["error"] = str(e)
   

StatementMeta(, 2b6f36c6-e2e2-4ee9-9034-7c2b0da3a180, 6, Finished, Available, Finished, False)

Оновлення довідника терміналів
Оновлення довідника терміналів-OK


### 2.3. Завантажуємо таблицю клієнтів

In [5]:
if not continue_processing :
    mssparkutils.notebook.exit(json.dumps(result_data))

try:

    # Виконую відбір клієнтів з транзакцій за дату 
    # і роблю   merge в таблицю клієнтів
    print("Оновлення таблиці клієнтів")
    
    # отримую параметри завантаження
    cu_up_id = result_data["upload_id"] 
    cu_up_opdate = result_data["upload_opdate"] 
    cu_idt_ts = datetime.now()

    cu_target_table="psh_dwh_lh.dbo.ev2_silver_customers_dim"
    cu_source_table="psh_dwh_lh.dbo.ev2_file_body"

    print("Оновлення таблиці клієнтів-формую буфер клінтів з транзакцій....")
    # формую dataframe клієнтів з таблиці  тіла файла по транзакіям за  cu_up_opdate

    cu_df_buff = spark.sql(f"""
            SELECT tr.NAME as cust_name, tr.PASSPORT as cust_doc
            FROM {cu_source_table} tr
            WHERE tr.OPDATE = :p_opdate
       """,
       {
        "p_opdate": cu_up_opdate
       }
    )
    print("Оновлення таблиці клієнтів-формую буфер клінтів з транзакцій - OK")
    # Коментую бо не потрібно
    # Додаю службові стовпці і заповнюю службові стовці spark dataframe значеннями
    #cu_df_buff = cu_df_buff.withColumn("cust_id", expr("uuid()")) \
    #                       .withColumn("cust_status", lit("ACTIVE")) \
    #                       .withColumn("upload_id", lit(cu_up_id)) \
    #                       .withColumn("idt", lit(cu_idt_ts))
    
    # роблю merge в target table psh_dwh_lh.dbo.ev2_silver_customers_dim
    print("Оновлення таблиці клієнтів - роблю merge.....")
    # 1. Спершу видаляємо дублікати з вхідного потоку (якщо один клієнт зробив 2 транзакції)
    # Залишаємо унікальні пари Name + Passport
    cu_df_distinct = cu_df_buff.select("cust_name", "cust_doc").distinct()

    # 2. Генеруємо UUID заздалегідь для всього набору
    # Тепер кожен РЯДОК у cu_df_final має свій фіксований ID
    cu_df_final = cu_df_distinct.withColumn("new_cust_id", expr("uuid()"))

    # 3. Отримуємо посилання на цільову Delta-таблицю
    delta_target = DeltaTable.forName(spark, cu_target_table)

    # 4. Виконуємо MERGE
    print("Оновлення таблиці клієнтів - виконання MERGE.....")
    delta_target.alias("target").merge(
        cu_df_final.alias("source"),
        "target.cust_name = source.cust_name AND target.cust_doc = source.cust_doc"
    ).whenMatchedUpdate(set = {
        "upload_id": lit(cu_up_id),
        "idt": lit(cu_idt_ts)
    }).whenNotMatchedInsert(values = {
        "cust_id": "source.new_cust_id", # Використовуємо вже згенерований ID
        "cust_name": "source.cust_name",
        "cust_doc": "source.cust_doc",
        "cust_status": lit("ACTIVE"),
        "upload_id": lit(cu_up_id),
        "idt": lit(cu_idt_ts)
    }).execute()

    print("Оновлення таблиці клієнтів - роблю merge - OK")
    print("Оновлення таблиці клієнтів-OK")
except Exception as e:
    print(f"Оновлення таблиці клієнтів-ERROR : {e}")
    continue_processing = False
    result_data["status"] = "Error"
    result_data["error"] = str(e)

StatementMeta(, 2b6f36c6-e2e2-4ee9-9034-7c2b0da3a180, 7, Finished, Available, Finished, False)

Оновлення таблиці клієнтів
Оновлення таблиці клієнтів-формую буфер клінтів з транзакцій....
Оновлення таблиці клієнтів-формую буфер клінтів з транзакцій - OK
Оновлення таблиці клієнтів - роблю merge.....
Оновлення таблиці клієнтів - виконання MERGE.....
Оновлення таблиці клієнтів - роблю merge - OK
Оновлення таблиці клієнтів-OK


## 2.4. Завантажуємо транзакції

In [6]:
if not continue_processing :
    mssparkutils.notebook.exit(json.dumps(result_data))

try:

    # Виконю merge з довідника терміналів
    print("Оновлення журналу транзакцій....")
    trn_up_id=result_data["upload_id"] 
    trn_up_opdate = result_data["upload_opdate"] 
    trn_idt_ts = datetime.now()
    target_table="psh_dwh_lh.dbo.ev2_silver_transactions"
    source_table="psh_dwh_lh.dbo.ev2_file_body"
    spark.sql(f"""
            MERGE INTO {target_table} AS target
            USING {source_table} AS source
            ON target.tx_id = source.TX_ID 
                AND target.file_id = source.FILE_ID 
                AND target.opdate = source.OPDATE
                AND source.OPDATE = :p_opdate
            WHEN MATCHED THEN
                UPDATE SET 
                    target.terminal_id = source.TERMINAL_ID, 
                    target.opdate = source.OPDATE,
                    target.amount = source.AMOUNT,
                    target.charge = source.CHARGE,
                    target.status = 'MODIFIED',
                    target.file_name = source.FILE_NAME,
                    target.file_id = source.FILE_ID,
                    target.upload_id=:trn_up_id,
                    target.idt = :idt_ts
            WHEN NOT MATCHED THEN
                INSERT (tx_id, terminal_id, opdate, amount, charge, status,file_name, file_id, upload_id, idt )
                VALUES (source.TX_ID, source.TERMINAL_ID, source.OPDATE, source.AMOUNT, source.CHARGE, 'ACCEPTED',
                         source.FILE_NAME, source.FILE_ID, :trn_up_id, :idt_ts)
        """
        , {
            "trn_up_id": trn_up_id, 
            "idt_ts": trn_idt_ts,
            "p_opdate": trn_up_opdate
        }
    )
    print("Оновлення журналу транзакцій - OK")
except Exception as e:
    print(f"Оновлення журналу транзакцій ERROR : {e}")
    continue_processing = False
    result_data["status"] = "Error"
    result_data["error"] = str(e)
   

StatementMeta(, 2b6f36c6-e2e2-4ee9-9034-7c2b0da3a180, 8, Finished, Available, Finished, False)

Оновлення журналу транзакцій....
Оновлення журналу транзакцій - OK


### 2.5. Фінішую процес завантаження

In [7]:

if not continue_processing :
    mssparkutils.notebook.exit(json.dumps(result_data))

try:
    
    fin_up_id = result_data["upload_id"] = up_id
    fin_up_opdate = result_data["upload_opdate"]
    print(f" Фінішую завантаження даних в Silver  {fin_up_id} за {fin_up_opdate} .....")
    close_upload(  fin_up_id )
    result_data["status"] = "SUCCESS"
    print(f" Фінішую завантаження даних в Silver  {fin_up_id} за {fin_up_opdate} - OK")
except Exception as e:
    print(f"Помилка обробки : {e}")
    continue_processing = False
    result_data["status"] = "Error"
    result_data["error"] = str(e)    

mssparkutils.notebook.exit(json.dumps(result_data))


StatementMeta(, 2b6f36c6-e2e2-4ee9-9034-7c2b0da3a180, 9, Finished, Available, Finished, False)

 Фінішую завантаження даних в Silver  c96f418c-b315-4bdd-90e0-ce40c73b0ac3 за 2025-04-15 .....
Фінішовано успішно процес завантаження c96f418c-b315-4bdd-90e0-ce40c73b0ac3 
 Фінішую завантаження даних в Silver  c96f418c-b315-4bdd-90e0-ce40c73b0ac3 за 2025-04-15 - OK
ExitValue: {"status": "SUCCESS", "error": null, "upload_id": "c96f418c-b315-4bdd-90e0-ce40c73b0ac3", "upload_opdate": "2025-04-15"}